# Velocity-only p(v | R) vs joint p(x, y, v) on the Draco mocks

Chains from `diagnostics/mock_sf_compare.py` for the twelve mocks (six with the Walker 2023 footprint, six without selection). Summary columns: set, mock, model, star count, true and posterior median log10 rho(150 pc) in Msun/kpc^3, 68% half-width (dex), z = (median - truth)/half-width, and the position-factor leak: std over the velocity-only posterior of log L_joint - log L_los (nats; zero for an exact theta-independent position factor) and its correlation with log rho(150). Figures: corner plots (seven labels, radii in kpc) and four-panel profiles (density, slope, sigma_los and kappa_los against the binned data and the truth).

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, os.path.expanduser('~/my_modules'))
os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
MODELS = Path(os.path.expanduser('~/scratch/projects/sbi_dsph/trained_models/nle/nle_8p_ZhaoPlumCOM_v4'))
OUT = Path(os.path.expanduser('~/scratch/projects/nle_dsph'))

In [ ]:
CHAINS = OUT / 'mock_sf'
df = pd.read_csv(CHAINS / 'mock_sf_compare.csv')
wide = df.pivot_table(index=['set', 'mock'], columns='model', values=['median', 'halfwidth', 'z'])
wide['hw_ratio'] = wide[('halfwidth', 'joint')] / wide[('halfwidth', 'los')]
display(wide.round(3)); display(df[df.model == 'joint'][['set', 'mock', 'leak_std_nats', 'leak_corr_rho150']].round(2))

In [ ]:
for f in sorted((CHAINS / 'figures').glob('profiles_*.png')):
    display(Image(str(f), width=1100))

In [ ]:
for f in sorted((CHAINS / 'figures').glob('corner_*.png')):
    display(Image(str(f), width=900))

Re-run one mock interactively (both models, short chain) and the leak diagnostic:

In [ ]:
import torch
from nle import NLE
from diagnostics import mock_sf_compare as msc
from diagnostics.pit_by_regime import best_checkpoint
import dsph_analysis.kinematic_io as kio
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
models = {'los': NLE.load_from_checkpoint(best_checkpoint(MODELS / '6xaw28uw'), map_location=dev, pre_transforms=None).eval(),
          'joint': NLE.load_from_checkpoint(best_checkpoint(sorted(MODELS.glob('*'))[0]), map_location=dev, pre_transforms=None).eval()}  # <- set the joint run id
meta = kio.load_meta('draco_1'); rhalf = float(meta.rhalf_kpc.value); lrs = float(np.log10(rhalf))
SET, PROF = 'noSF', 'CoreOM'
obs = msc.load_mock(msc.SETS[SET], PROF, meta)
params = pd.read_csv(msc.MOCKS / msc.SETS[SET] / 'mock_catalog_params.csv')
truth = msc.truth_labels(params[params.profile == PROF].iloc[0], rhalf)
chains = {k: msc.sample(m, obs, lrs, 48, 800, 0, dev) for k, m in models.items()}
for k, c in chains.items():
    lr = msc.log_rho_150(c[:, :7], lrs); print(k, np.percentile(lr, [16, 50, 84]).round(3), 'truth', msc.log_rho_150(truth[None], lrs).round(3))
print('leak std [nats], corr with log rho150:', msc.position_factor_leak(models, obs, lrs, chains['los'], dev))